In [ ]:
# ============================================================================
# 03a — SETUP + MODEL CHECK + PROVIDER PINNING
# ----------------------------------------------------------------------------
# Runs in five stages:
#   1. load benchmark, prompts, parser
#   2. ID CHECK   — one call per model, catches typos before they cost money
#   3. PIN PROBE  — for multi-provider models, find one that can serve the run
#   4. PIN APPLY  — rewire call() so every model goes to a fixed provider
#   5. VERIFY     — confirm the pins hold and flag anything already collected
# Total cost: a few cents.
# ============================================================================
import os, json, time, re, threading, hashlib, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd, numpy as np
from dotenv import load_dotenv

ROOT = Path.cwd().parent
load_dotenv(ROOT/".env", override=True)
KEY = os.environ.get("OPENROUTER_API_KEY", "")
assert len(KEY) > 20, "OPENROUTER_API_KEY missing from .env"
ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"

PROC, RES = ROOT/"data"/"processed", ROOT/"results"
RES.mkdir(parents=True, exist_ok=True)
VERSION = "v2"

# ============================ 1. BENCHMARK ================================
FULL  = pd.read_json(PROC/f"sighabit_{VERSION}_full.jsonl", lines=True)
USUAL = pd.read_json(PROC/f"sighabit_{VERSION}_usually.jsonl", lines=True)
PR    = json.loads((PROC/f"prompts_{VERSION}.json").read_text())
SHA   = hashlib.sha256((PROC/f"sighabit_{VERSION}_full.jsonl").read_bytes()).hexdigest()[:16]
MAIN_ONLY = FULL[~FULL.is_decoy].reset_index(drop=True)   # swap arm
WEB_ITEMS = MAIN_ONLY                                     # web arm (no decoys: search fees)
print(f"benchmark {len(FULL)} main+decoy | {len(USUAL)} usually | sha={SHA}")
print(f"  {dict(FULL.condition.value_counts())}")

def render(item, swap=False):
    if item.get("condition") == "usually":
        body = (f"Across its known direct targets, what is the net regulatory "
                f"effect of {item['source']} at the protein level?\n{PR['guide_usually']}")
    else:
        body = (f"What is the net regulatory effect of {item['source']} on "
                f"{item['target']} at the protein level?\n{PR['guide']}")
    return f"{body}\n\nRespond with JSON only:\n" + (
        PR['schema_swapped'] if swap else PR['schema'])

# ============================ PARSER ======================================
SYN = {"stimulates":"stimulates","stimulate":"stimulates","activates":"stimulates",
 "activate":"stimulates","activation":"stimulates","positive":"stimulates",
 "upregulates":"stimulates","induces":"stimulates",
 "inhibits":"inhibits","inhibit":"inhibits","inhibition":"inhibits",
 "represses":"inhibits","repress":"inhibits","negative":"inhibits",
 "downregulates":"inhibits","suppresses":"inhibits",
 "not_established":"not_established","not established":"not_established",
 "unknown":"not_established","none":"not_established","unclear":"not_established"}

def strip_reasoning(t):
    """Reasoning models restate the answer template in their scratchpad;
    strip it or we parse the example instead of the answer."""
    t = re.sub(r"<think>.*?</think>", " ", t, flags=re.S|re.I)
    t = re.sub(r"^.*?</think>", " ", t, flags=re.S|re.I)
    return t.strip() or t

def parse(text):
    if not isinstance(text,str) or not text.strip(): return None,"failed"
    b = strip_reasoning(text)
    try:
        o = json.loads(b)
        if isinstance(o,dict): return o,"strict"
    except Exception: pass
    c = re.sub(r"```(?:json)?|```", " ", b)
    for blob in reversed(re.findall(r"\{[^{}]*\}", c, flags=re.S)):
        try:
            o = json.loads(re.sub(r",\s*([}\]])", r"\1", blob))
            if isinstance(o,dict) and "answer" in o: return o,"repaired"
        except Exception: continue
    m = re.search(r'"?answer"?\s*[:=]\s*"?([A-Za-z_ ]+?)"?\s*[,\n}]', c, re.I)
    if m:
        out = {"answer": m.group(1).strip()}
        cf = re.search(r'"?confidence"?\s*[:=]\s*(\d+)', c, re.I)
        if cf: out["confidence"] = int(cf.group(1))
        return out,"regex"
    return None,"failed"

def normalize(p):
    if not p or "answer" not in p: return None
    return SYN.get(str(p["answer"]).strip().lower().replace("-","_"))

for txt, want in [('{"answer":"stimulates","confidence":8}',"stimulates"),
                  ('```json\n{"answer":"inhibits",}\n```',"inhibits"),
                  ('<think>maybe stimulates</think>{"answer":"inhibits"}',"inhibits"),
                  ('{"answer":"activates"}',"stimulates"),
                  ('{"answer":"not_established"}',"not_established"),
                  ('I think it activates.',None)]:
    assert normalize(parse(txt)[0]) == want, f"parser failed on {txt!r}"
print("parser self-test: 6/6 pass")

# ============================ MODELS ======================================
# name, openrouter id, $/M in, $/M out, max tokens, candidate providers
# Empty provider list = single-vendor (OpenAI/Google/Anthropic serve their own).
MODELS_SPEC = [
 ("gpt-oss-120b",     "openai/gpt-oss-120b",               0.03,  0.17,  6000,
    ["DeepInfra","CoreWeave","Novita","Parasail"]),
 ("qwen3-32b",        "qwen/qwen3-32b",                    0.10,  0.30,  4000,
    ["DeepInfra","Nebius","SiliconFlow","Groq"]),
 ("gemini-3.7-flash", "google/gemini-3.7-flash",           0.1875,0.9375,3000, []),
 ("gpt-5.6-luna",     "openai/gpt-5.6-luna",               0.10,  0.60,  5000, []),
 ("gpt-5.6-terra",    "openai/gpt-5.6-terra",              1.00,  6.00,  5000, []),
 # --- new: other labs, other training data ---
 ("claude-sonnet-4.5","anthropic/claude-sonnet-4.5",       3.00, 15.00,  4000, []),
 ("claude-haiku-4.5", "anthropic/claude-haiku-4.5",        1.00,  5.00,  4000, []),
 ("llama-3.3-70b",    "meta-llama/llama-3.3-70b-instruct", 0.12,  0.30,  3000,
    ["DeepInfra","Novita","Nebius","Together"]),
 ("mistral-large",    "mistralai/mistral-large",           2.00,  6.00,  3000, []),
 ("deepseek-v3",      "deepseek/deepseek-chat",            0.14,  0.28,  4000,
    ["DeepInfra","Novita","Together","Fireworks"]),
 ("gemma-3-27b",      "google/gemma-3-27b-it",             0.10,  0.20,  3000,
    ["DeepInfra","Novita","Nebius"]),
 ("qwen3-235b",       "qwen/qwen3-235b-a22b",              0.20,  0.60,  4000,
    ["DeepInfra","Nebius","Together"]),
]
BUDGET, WORKERS, TEMP = 8.00, 6, 0.3
SPEND = {"usd":0.0}; STOP = threading.Event(); LOCK = threading.Lock()

def _post(body, timeout):
    r = requests.post(ENDPOINT, headers={"Authorization":f"Bearer {KEY}"},
                      json=body, timeout=timeout)
    return r

# ==================== 2. ID CHECK — do these models exist? ================
print("\n"+"="*88); print("2. MODEL ID CHECK  (1 call each)"); print("="*88)
print("A wrong id fails every call. Catch it now for pennies.\n")
alive = []
for name, mid, pi, po, mt, cands in MODELS_SPEC:
    body = {"model":mid, "temperature":0, "max_tokens":32,
            "messages":[{"role":"user","content":"Reply with the single word: ok"}]}
    try:
        r = _post(body, 90)
        if r.status_code == 200:
            d = r.json()
            txt = (d["choices"][0]["message"].get("content") or "").strip()[:20]
            print(f"  [OK ] {name:<20}{mid:<38}{d.get('provider','?'):<14}{txt!r}")
            alive.append((name, mid, pi, po, mt, cands))
        else:
            print(f"  [BAD] {name:<20}{mid:<38}HTTP {r.status_code}  "
                  f"{r.text[:70]}")
    except Exception as e:
        print(f"  [BAD] {name:<20}{mid:<38}{type(e).__name__}")
dropped = [m[0] for m in MODELS_SPEC if m[0] not in {a[0] for a in alive}]
print(f"\n  usable: {len(alive)}/{len(MODELS_SPEC)}"
      + (f"   DROPPED: {dropped}" if dropped else ""))
print("  Fix any bad ids at openrouter.ai/models and re-run this cell.")

# ==================== 3. PIN PROBE =======================================
print("\n"+"="*88); print("3. PROVIDER PROBE  (8 calls per candidate)"); print("="*88)
print("Open-weight models can be served by many providers at different")
print("quantizations. Pin each to one, so a mid-run switch cannot move results.\n")
PROBE_N = 8
probe_items = FULL.head(PROBE_N)
PIN = {}
for name, mid, pi, po, mt, cands in alive:
    if not cands:
        PIN[name] = None
        print(f"  {name:<20}single-vendor — nothing to pin"); continue
    print(f"  {name}")
    for cand in cands:
        good, provs, t0 = 0, set(), time.time()
        for _, it in probe_items.iterrows():
            try:
                b = {"model":mid, "temperature":TEMP, "max_tokens":mt,
                     "messages":[{"role":"system","content":PR["system"]},
                                 {"role":"user","content":render(it.to_dict())}],
                     "provider":{"order":[cand], "allow_fallbacks":False}}
                r = _post(b, 120)
                if r.status_code == 200:
                    d = r.json()
                    good += bool(d["choices"][0]["message"].get("content"))
                    if d.get("provider"): provs.add(d["provider"])
            except Exception: pass
        honoured = (provs <= {cand}) or not provs
        print(f"    {cand:<14}{good}/{PROBE_N}  {good/PROBE_N:>5.0%}  "
              f"{(time.time()-t0)/PROBE_N:>5.1f}s/call  got={provs or '?'}"
              + ("" if honoured else "   *** PIN IGNORED ***"))
        if good/PROBE_N >= 0.875 and honoured:
            PIN[name] = cand; print(f"    --> PINNED to {cand}"); break
    else:
        PIN[name] = None
        print(f"    --> no reliable pin; leaving unpinned "
              f"(provider logged per call, so still auditable)")

# ==================== 4. PIN APPLY =======================================
MODELS = [(n,m,pi,po,mt) for n,m,pi,po,mt,_ in alive]
_MID2PIN = {m: PIN.get(n) for n,m,*_ in MODELS}

def call(mid, user, max_tok, web=False, timeout=180):
    """One question, one call. Provider pinned where a pin exists. Empty
    completions are retried — a silent gap is worse than a slow retry."""
    body = {"model":mid, "temperature":TEMP, "max_tokens":max_tok,
            "messages":[{"role":"system","content":PR["system"]},
                        {"role":"user","content":user}]}
    if web: body["plugins"] = [{"id":"web","max_results":3}]
    pin = _MID2PIN.get(mid)
    if pin: body["provider"] = {"order":[pin], "allow_fallbacks":False}
    last_empty = None
    for att in range(5):
        if STOP.is_set():
            return dict(text=None,usage={},http=0,error="stopped",prov=None)
        try:
            r = _post(body, timeout)
            if r.status_code == 200:
                d = r.json(); ch = d["choices"][0]
                txt = ch["message"].get("content")
                if not txt:
                    last_empty = dict(text=None, usage=d.get("usage",{}), http=204,
                                      error=f"empty ({ch.get('finish_reason')})",
                                      prov=d.get("provider"))
                    time.sleep(2+att); continue
                return dict(text=txt, usage=d.get("usage",{}), http=200,
                            error=None, prov=d.get("provider"))
            if r.status_code in (429,503,529):
                time.sleep(min(5*(1.8**att),60)); continue
            if r.status_code in (401,402,403):
                STOP.set()
                return dict(text=None,usage={},http=r.status_code,
                            error=r.text[:200],prov=None)
            return dict(text=None,usage={},http=r.status_code,
                        error=r.text[:200],prov=None)
        except Exception:
            time.sleep(min(2**att,20))
    return last_empty or dict(text=None,usage={},http=0,
                              error="retries exhausted",prov=None)

def run(name, mid, pin_price, pout, max_tok, items, arm, swap=False, web=False):
    """Ask one model every question in `items`. Writes every attempt to disk."""
    tag  = f"{VERSION}__{name}__{arm}"
    path = RES/f"{tag}.jsonl"
    done = set()
    if path.exists():
        rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
        done = {r["item_id"] for r in rows if r.get("http")==200 and r.get("raw")}
    todo = items[~items.item_id.isin(done)]
    if not len(todo):
        print(f"  {tag}: complete ({len(done)})"); return
    print(f"  {tag}: {len(todo)} to do", flush=True)
    t0, n, bad = time.time(), [0], [0]
    def one(d, f):
        if STOP.is_set(): return
        res = call(mid, render(d, swap), max_tok, web)
        parsed, tier = parse(res["text"]) if res["http"]==200 else (None,"failed")
        u = res.get("usage") or {}
        cost = (u.get("prompt_tokens",0)/1e6*pin_price
                + u.get("completion_tokens",0)/1e6*pout)
        rec = dict(item_id=d["item_id"], benchmark_sha=SHA, model=name, arm=arm,
            model_id=mid, provider=res["prov"], pinned=_MID2PIN.get(mid),
            swap=bool(swap), web=bool(web),
            source=d["source"], target=d.get("target"),
            condition=d["condition"], truth=d["answer"],
            protein_usually=d.get("protein_usually"), group=d.get("group"),
            n_refs=d.get("n_refs"), is_decoy=bool(d.get("is_decoy", False)),
            pred=normalize(parsed), confidence=(parsed or {}).get("confidence"),
            rationale=(parsed or {}).get("rationale"), parse_tier=tier,
            raw=res["text"], prompt_tokens=u.get("prompt_tokens"),
            completion_tokens=u.get("completion_tokens"), cost_usd=cost,
            http=res["http"], error=res["error"])
        with LOCK:
            SPEND["usd"] += cost; n[0] += 1
            if res["http"] != 200: bad[0] += 1
            f.write(json.dumps(rec)+"\n"); f.flush()
            if SPEND["usd"] > BUDGET:
                STOP.set(); print(f"\n*** BUDGET ${BUDGET} REACHED — STOPPING ***")
            if n[0] % max(1,len(todo)//5) == 0 or n[0]==len(todo):
                print(f"    {n[0]}/{len(todo)}  {(time.time()-t0)/60:.1f}m  "
                      f"fail {bad[0]}  ${SPEND['usd']:.3f}", flush=True)
    with open(path,"a",encoding="utf-8") as f:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            list(ex.map(lambda r: one(r[1].to_dict(), f), todo.iterrows()))
    print(f"    done {(time.time()-t0)/60:.1f}m  failures {bad[0]}/{len(todo)}")

json.dump(PIN, open(PROC/"provider_pins.json","w"), indent=2)

# ==================== 5. VERIFY ==========================================
print("\n"+"="*88); print("5. VERIFY THE PINS HOLD"); print("="*88)
bad_pin = 0
for name, mid, pi, po, mt in MODELS:
    want, got = PIN.get(name), set()
    for _, it in FULL.head(3).iterrows():
        r = call(mid, render(it.to_dict()), mt, timeout=120)
        if r["prov"]: got.add(r["prov"])
    if want:
        ok = got <= {want}
        bad_pin += (not ok)
        print(f"  [{'OK ' if ok else 'BAD'}] {name:<20}want={want:<14}got={got}")
    else:
        print(f"  [ -- ] {name:<20}unpinned{'':<10}got={got}"
              + ("" if len(got)<=1 else "   *** MULTI-PROVIDER — disclose ***"))
assert bad_pin == 0, "a pin was not honoured"

print("\n"+"="*88); print("ALREADY COLLECTED?"); print("="*88)
have = list(RES.glob("*.jsonl"))
if have:
    A = pd.concat([pd.read_json(p, lines=True) for p in have], ignore_index=True)
    A = A[A.http==200]
    print(f"  {len(A):,} rows on disk, {A.model.nunique()} models")
    stale = []
    for m, g in A.groupby("model"):
        want = PIN.get(m); provs = set(g.provider.dropna())
        flag = ""
        if want and provs - {want}: flag = f"  *** not all {want} ***"; stale.append(m)
        elif len(provs) > 1: flag = "  *** multiple providers ***"
        print(f"    {m:<20}{dict(g.provider.value_counts())}{flag}")
    if stale:
        print(f"\n  wrong-provider rows for {stale} — delete those files to recollect")
    todo_models = [n for n,*_ in MODELS if n not in set(A.model)]
    print(f"\n  models still to collect: {todo_models}")
else:
    print("  results folder empty")

# ---- projection -----------------------------------------------------------
print("\n"+"="*88); print("PLAN"); print("="*88)
n_calls = len(FULL) + len(USUAL)
tot = 0
print(f"{'model':<20}{'pin':<14}{'main+usually $':>16}{'+swap $':>10}")
print("-"*62)
for name, mid, pi, po, mt in MODELS:
    c = n_calls*(160/1e6*pi + 350/1e6*po)
    s = len(MAIN_ONLY)*(160/1e6*pi + 350/1e6*po)
    tot += c + s
    print(f"{name:<20}{str(PIN.get(name) or '—'):<14}{c:>16.3f}{s:>10.3f}")
web = len(WEB_ITEMS)*3*4/1000 + len(WEB_ITEMS)*(160/1e6*.1875 + 480/1e6*.9375)
print(f"\n  all models (new work only will actually be charged): ${tot:.2f}")
print(f"  web arm (already collected)                        : ${web:.2f}")
print(f"  budget ${BUDGET}   |   spent so far this session ${SPEND['usd']:.3f}")
print("\nReady. Run 03b (preflight) then 03c (collection).")

benchmark 336 main+decoy | 108 usually | sha=99be1e9522e4f933
  {'decoy': np.int64(120), 'habit_agrees': np.int64(108), 'habit_misleads': np.int64(108)}
parser self-test: 6/6 pass

2. MODEL ID CHECK  (1 call each)
A wrong id fails every call. Catch it now for pennies.

  [OK ] gpt-oss-120b        openai/gpt-oss-120b                   DeepInfra     ''
  [OK ] qwen3-32b           qwen/qwen3-32b                        DeepInfra     ''
  [OK ] gemini-3.7-flash    google/gemini-3.7-flash               Google        ''
  [OK ] gpt-5.6-luna        openai/gpt-5.6-luna                   OpenAI        'ok'
  [OK ] gpt-5.6-terra       openai/gpt-5.6-terra                  OpenAI        'ok'
  [OK ] claude-sonnet-4.5   anthropic/claude-sonnet-4.5           Amazon Bedrock'ok'
  [OK ] claude-haiku-4.5    anthropic/claude-haiku-4.5            Amazon Bedrock'ok'


In [ ]:
# # ============================================================================
# # 03b — PREFLIGHT  (~2 cents; verifies everything before the real run)
# # ============================================================================
# MINI  = pd.concat([FULL[FULL.condition==c].head(3)
#                    for c in ["habit_agrees","habit_misleads","decoy"]]).reset_index(drop=True)
# MINIU = USUAL.head(3)
# print(f"preflight: {len(MINI)} main + {len(MINIU)} usually per model "
#       f"= ~{(len(MINI)+len(MINIU))*len(MODELS)+8} calls\n")

# SPEND["usd"] = 0.0; STOP.clear()
# rows = []

# def probe(name, mid, pin, pout, mt, item, swap=False, web=False):
#     t0 = time.time()
#     res = call(mid, render(item, swap), mt, web, timeout=180)
#     parsed, tier = parse(res["text"]) if res["http"]==200 else (None,"failed")
#     pred = normalize(parsed)
#     u = res.get("usage") or {}
#     cost = u.get("prompt_tokens",0)/1e6*pin + u.get("completion_tokens",0)/1e6*pout
#     with LOCK: SPEND["usd"] += cost
#     return dict(model=name, cond=item["condition"], swap=swap, web=web,
#         source=item["source"], target=item.get("target"), truth=item["answer"],
#         pred=pred, correct=(pred==item["answer"]),
#         conf=(parsed or {}).get("confidence"), tier=tier,
#         rationale=(parsed or {}).get("rationale"),
#         raw=(res["text"] or "")[:200], http=res["http"], err=res["error"],
#         prov=res["prov"], out_tok=u.get("completion_tokens"),
#         secs=time.time()-t0, cost=cost)

# print("="*96); print("A. EVERY MODEL, EVERY QUESTION TYPE"); print("="*96)
# for name, mid, pin, pout, mt in MODELS:
#     print(f"\n{name}", flush=True)
#     for _, it in pd.concat([MINI, MINIU]).iterrows():
#         r = probe(name, mid, pin, pout, mt, it.to_dict()); rows.append(r)
#         ok = r["http"]==200 and r["pred"]
#         print(f"  [{'OK ' if ok else 'BAD'}] {r['cond']:<15}"
#               f"{str(r['source']):>9}->{str(r['target'] or '—'):<9} "
#               f"truth={str(r['truth']):<16} pred={str(r['pred']):<16}"
#               f"{r['tier']:<9}{r['secs']:>5.1f}s"
#               + (f"  ERR {str(r['err'])[:55]}" if r["http"]!=200 else ""))

# print("\n"+"="*96); print("B. ORDER-SWAP — does flipping the options change the answer?")
# print("="*96)
# for name, mid, pin, pout, mt in MODELS[:3]:
#     it = MINI[MINI.condition=="habit_agrees"].iloc[0].to_dict()
#     a = probe(name, mid, pin, pout, mt, it, swap=False)
#     b = probe(name, mid, pin, pout, mt, it, swap=True)
#     rows += [a,b]
#     print(f"  {name:<20} normal={str(a['pred']):<14} swapped={str(b['pred']):<14}"
#           f"  {'same' if a['pred']==b['pred'] else 'DIFFERENT'}")

# print("\n"+"="*96); print("C. WEB SEARCH — is retrieval actually happening?"); print("="*96)
# for _, it in MINI.head(2).iterrows():
#     r = probe("gemini-3.7-flash","google/gemini-3.7-flash",.1875,.9375,3000,
#               it.to_dict(), web=True); rows.append(r)
#     print(f"  [{'OK ' if r['http']==200 else 'BAD'}] {r['cond']:<15}"
#           f"{r['source']}->{r['target']}  pred={str(r['pred']):<16}"
#           f"{r['secs']:>5.1f}s  ${r['cost']:.4f}  {r['out_tok']} tok")
#     print(f"        {str(r['rationale'])[:130]}")

# T  = pd.DataFrame(rows)
# ok = T[T.http==200]
# print("\n"+"="*96); print("D. DID ANYTHING BREAK?"); print("="*96)
# checks = [
#  ("every model responded", T.groupby('model').http.apply(lambda s:(s==200).any()).all()),
#  ("no HTTP failures",      (T.http==200).all()),
#  ("everything parsed",     ok.pred.notna().all()),
#  ("decoys answered",       len(ok[ok.cond=='decoy'])>0),
#  ("'usually' probes work", len(ok[ok.cond=='usually'])>0),
#  ("swap arm works",        ok[ok.swap].pred.notna().all() if ok.swap.any() else True),
#  ("web arm works",         ok[ok.web].pred.notna().all() if ok.web.any() else True),
# ]
# for n_,c in checks: print(f"  [{'PASS' if c else 'FAIL'}] {n_}")
# if (T.http!=200).any():
#     print("\n  FAILURES:")
#     print("  "+T[T.http!=200][["model","cond","http","err"]].to_string(index=False).replace("\n","\n  "))
# if ok.pred.isna().any():
#     print("\n  UNPARSED — raw text:")
#     for _, r in ok[ok.pred.isna()].head(3).iterrows():
#         print(f"    {r.model}/{r.cond}: {r.raw!r}")

# print("\n"+"="*96); print("E. SANITY (3 items each — machinery check, not a result)")
# print("="*96)
# print(ok.pivot_table(index="model", columns="cond", values="correct",
#                      aggfunc="mean").round(2).to_string())
# d = ok[ok.cond=="decoy"]
# print("\ndecoy refusal rate (some models should be >0):")
# print("  "+(d.assign(ref=d.pred=="not_established").groupby("model").ref.mean()
#       .round(2).to_string().replace("\n","\n  ")))

# print("\n"+"="*96); print("F. READ THESE RATIONALES"); print("="*96)
# for m in ok.model.unique():
#     s = ok[(ok.model==m)&(ok.cond=="habit_misleads")&(~ok.web)]
#     if not len(s): continue
#     r = s.iloc[0]
#     print(f"\n  {m}  |  {r.source} -> {r.target}  (truth: {r.truth})")
#     print(f"    said {r.pred}, confidence {r.conf}")
#     print(f"    {str(r.rationale)[:150]}")

# print("\n"+"="*96); print("G. COST PROJECTION"); print("="*96)
# print(f"  preflight cost: ${SPEND['usd']:.4f} over {len(T)} calls\n")
# per = ok[~ok.web].groupby("model").agg(secs=("secs","mean"), cost=("cost","mean"),
#                                        out=("out_tok","mean"))
# n_calls = len(FULL)+len(USUAL)
# print(f"{'model':<20}{'s/call':>8}{'out tok':>9}{'$/call':>9}{'full run $':>12}{'min':>7}")
# print("-"*66)
# tot = 0
# for m in per.index:
#     c = per.loc[m,"cost"]*n_calls; tot += c
#     print(f"{m:<20}{per.loc[m,'secs']:>8.1f}{per.loc[m,'out']:>9.0f}"
#           f"{per.loc[m,'cost']:>9.4f}{c:>12.3f}"
#           f"{per.loc[m,'secs']*n_calls/WORKERS/60:>7.0f}")
# web_cost = len(FULL)*3*4/1000 + len(FULL)*(160/1e6*.1875 + 400/1e6*.9375)
# print(f"\n  main arms        : ${tot:.2f}")
# print(f"  swap (3 models)  : ${tot*0.13:.2f}")
# print(f"  web (Gemini, x3) : ${web_cost:.2f}")
# print(f"  PROJECTED TOTAL  : ${tot + tot*0.13 + web_cost:.2f}   (budget ${BUDGET})")

# good = all(c for _,c in checks)
# print("\n"+"="*96)
# print("ALL CHECKS PASSED — cell 03c is safe to run" if good
#       else "*** SOMETHING FAILED — do not run 03c ***")
# print("="*96)

preflight: 9 main + 3 usually per model = ~68 calls

A. EVERY MODEL, EVERY QUESTION TYPE

gpt-oss-120b
  [OK ] habit_agrees       NFIL3->PTGS2     truth=inhibits         pred=inhibits        strict    18.7s
  [OK ] habit_agrees        DOK1->ITGB2     truth=inhibits         pred=not_established strict     5.3s
  [OK ] habit_agrees       HDAC1->ABCB1     truth=inhibits         pred=inhibits        strict     2.2s
  [OK ] habit_misleads     NFIL3->IL3       truth=stimulates       pred=inhibits        strict     3.1s
  [OK ] habit_misleads      DOK1->RASGRP3   truth=stimulates       pred=not_established strict     3.9s
  [OK ] habit_misleads     HDAC1->COL1A2    truth=stimulates       pred=inhibits        strict     6.9s
  [OK ] decoy             DNMT3B->OXT       truth=not_established  pred=not_established strict     3.1s
  [OK ] decoy             DNMT3B->STAT3     truth=not_established  pred=not_established strict     9.7s
  [OK ] decoy              MAPK7->HIPK2     truth=not_established

In [ ]:
# ============================================================================
# 03c — FULL COLLECTION   *** SPENDS MONEY ***
# ----------------------------------------------------------------------------
# Uses the patched `call` (retries empty responses) and WEB_ITEMS (216 real
# questions, no decoys) to stay inside budget. Resumable: safe to interrupt.
# ============================================================================
assert "WEB_ITEMS" in globals(), "run the patch cell first"
SPEND["usd"] = 0.0; STOP.clear()
RUN_T0 = time.time()

print("="*70); print("MAIN ARM — 5 models x (336 questions + 108 usually)")
print("="*70)
for name, mid, pin, pout, mt in MODELS:
    if STOP.is_set(): print("\n*** HALTED ***"); break
    print(f"\n{name}", flush=True)
    run(name, mid, pin, pout, mt, FULL,  "main")
    run(name, mid, pin, pout, mt, USUAL, "usually")


print("\n"+"="*70); print("ORDER-SWAP ARM — all models"); print("="*70)
for name, mid, pin_, pout, mt in MODELS:          # was MODELS[:3]
    if STOP.is_set(): break
    run(name, mid, pin_, pout, mt, MAIN_ONLY, "swap", swap=True)
print("\n"+"="*70); print("WEB-SEARCH ARM — Gemini x 216"); print("="*70)
if not STOP.is_set():
    run("gemini-3.7-flash","google/gemini-3.7-flash",.1875,.9375,3000,
        WEB_ITEMS, "web", web=True)

# ---- inventory -----------------------------------------------------------
print("\n"+"="*70); print("INVENTORY"); print("="*70)
A = pd.concat([pd.read_json(p, lines=True) for p in RES.glob("*.jsonl")],
              ignore_index=True)
print(f"rows {len(A):,}   spent ${SPEND['usd']:.2f}   "
      f"wall clock {(time.time()-RUN_T0)/60:.0f} min")
print(f"http failures {(A.http!=200).mean():.2%}   "
      f"unparsed {(A[A.http==200].pred.isna()).mean():.2%}")

print("\nunique items per model x arm:")
print(A[A.http==200].pivot_table(index="model", columns="arm",
      values="item_id", aggfunc="nunique", fill_value=0).to_string())

print("\nEXPECTED:  main 336 | usually 108 | swap 216 (all 5 models) | web 216 (Gemini)")



if (A.http!=200).any():
    print("\nfailures by model and status:")
    print(A[A.http!=200].groupby(["model","http"]).size().to_string())
if (A[A.http==200].pred.isna()).any():
    u = A[(A.http==200) & A.pred.isna()]
    print(f"\nunparsed ({len(u)} rows):")
    print(u.groupby("model").size().to_string())
    for _, r in u.head(3).iterrows():
        print(f"    {r.model}: {str(r.raw)[:120]!r}")

print("\nspend by model:")
print(A.groupby("model").cost_usd.sum().round(3).to_string())

# ---- first look at the headline ------------------------------------------
print("\n"+"="*70); print("FIRST LOOK  (main arm, real questions only)"); print("="*70)
M = A[(A.arm=="main") & (A.http==200) & (~A.is_decoy) & A.pred.notna()]
if len(M):
    t = M.assign(ok=M.pred==M.truth).pivot_table(
        index="model", columns="condition", values="ok", aggfunc="mean")
    if {"habit_agrees","habit_misleads"} <= set(t.columns):
        t["gap_pp"] = ((t.habit_agrees - t.habit_misleads)*100).round(1)
        print((t[["habit_agrees","habit_misleads"]]*100).round(1)
              .assign(gap_pp=t.gap_pp).to_string())
        print("\n  gap = how much worse models do when the protein does the")
        print("  opposite of its usual thing. This is the whole study.")

MAIN ARM — 5 models x (336 questions + 108 usually)

gpt-oss-120b
  v2__gpt-oss-120b__main: complete (336)
  v2__gpt-oss-120b__usually: complete (108)

qwen3-32b
  v2__qwen3-32b__main: complete (336)
  v2__qwen3-32b__usually: complete (108)

gemini-3.7-flash
  v2__gemini-3.7-flash__main: complete (336)
  v2__gemini-3.7-flash__usually: complete (108)

gpt-5.6-luna
  v2__gpt-5.6-luna__main: complete (336)
  v2__gpt-5.6-luna__usually: complete (108)

gpt-5.6-terra
  v2__gpt-5.6-terra__main: complete (336)
  v2__gpt-5.6-terra__usually: complete (108)

ORDER-SWAP ARM — all 5 models
  v2__gpt-oss-120b__swap: complete (216)
  v2__qwen3-32b__swap: complete (216)
  v2__gemini-3.7-flash__swap: complete (216)
  v2__gpt-5.6-luna__swap: complete (216)
  v2__gpt-5.6-terra__swap: complete (216)

WEB-SEARCH ARM — Gemini x 216
  v2__gemini-3.7-flash__web: complete (216)

INVENTORY
rows 3,516   spent $0.00   wall clock 0 min
http failures 0.00%   unparsed 0.00%

unique items per model x arm:
arm        

In [10]:
# Confirms the patched call() is live and the web arm is trimmed.
import inspect
src = inspect.getsource(call)
print("patched call() active:", "last_empty" in src)          # must be True
print("WEB_ITEMS defined    :", "WEB_ITEMS" in globals(),
      f"({len(WEB_ITEMS)} items)" if "WEB_ITEMS" in globals() else "")
print("no decoys in web arm :", not WEB_ITEMS.is_decoy.any())
print("budget               :", BUDGET, "| spent so far:", round(SPEND["usd"],3))

patched call() active: True
WEB_ITEMS defined    : True (216 items)
no decoys in web arm : True
budget               : 5.0 | spent so far: 0.001


In [7]:
# ============================================================================
# 03-TRUTH — spot-check the ground truth before committing
# ----------------------------------------------------------------------------
# Every "habit_misleads" item claims a protein does the OPPOSITE of its usual
# behaviour. Those are exactly the records most likely to be curation errors.
# If they are wrong, our effect is measurement error, not model behaviour.
# This pulls the supporting PubMed IDs so we can check a sample by hand.
# ============================================================================
import urllib.request, urllib.parse
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
RAW  = ROOT/"data"/"raw"

# find the references behind each benchmark edge
ref_map = {}
for n in ["causal_ppi","collectri","kinaseextra"]:
    d = pd.read_parquet(RAW/f"{n}.parquet")
    col = "references" if "references" in d.columns else None
    if col is None: continue
    for s,t,r,src in zip(d.source_genesymbol, d.target_genesymbol,
                         d[col], d.get("sources", d[col])):
        if isinstance(r,str) and r.strip():
            ref_map.setdefault((s,t), set()).update(
                x.split(":")[-1] for x in r.split(";") if x.strip())

B = pd.read_json(ROOT/"data"/"processed"/"sighabit_v2.jsonl", lines=True)
B["pmids"] = [sorted(ref_map.get((r.source, r.target), set()))[:4]
              for _, r in B.iterrows()]
B["n_pmid"] = B.pmids.map(len)
print(f"items with at least one PubMed ID: {(B.n_pmid>0).mean():.0%}")

print("\n" + "="*92)
print("20 'HABIT MISLEADS' ITEMS TO VERIFY BY HAND")
print("="*92)
print("For each: does the cited paper really show this direction of effect?\n")
rng = np.random.RandomState(0)
S = B[(B.condition=="habit_misleads") & (B.n_pmid>0)]
S = S.iloc[rng.permutation(len(S))[:20]]
for i, (_, r) in enumerate(S.iterrows(), 1):
    print(f"{i:>3}. {r.source} -> {r.target}   OmniPath says: {r.answer.upper()}")
    print(f"     ({r.source} usually {r.protein_usually}; {r.n_refs} refs; {r.component})")
    for p in r.pmids[:3]:
        print(f"     https://pubmed.ncbi.nlm.nih.gov/{p}/")
    print()

print("="*92)
print("SAME FOR 'HABIT AGREES' — so we check both conditions equally")
print("="*92)
S2 = B[(B.condition=="habit_agrees") & (B.n_pmid>0)]
S2 = S2.iloc[rng.permutation(len(S2))[:10]]
for i, (_, r) in enumerate(S2.iterrows(), 1):
    print(f"{i:>3}. {r.source} -> {r.target}   OmniPath says: {r.answer.upper()}"
          f"   ({r.n_refs} refs)")
    for p in r.pmids[:2]:
        print(f"     https://pubmed.ncbi.nlm.nih.gov/{p}/")

B.to_json(ROOT/"data"/"processed"/"sighabit_v2_with_pmids.jsonl",
          orient="records", lines=True)
print(f"\nwritten: sighabit_v2_with_pmids.jsonl")
print("\nHOW TO USE THIS:")
print("  Open ~10 of the 'habit misleads' links and read the abstract.")
print("  Mark each: CORRECT / WRONG / AMBIGUOUS.")
print("  If 8+ of 10 are correct, the labels are sound and we proceed.")
print("  If 3+ are wrong, we must audit the whole set before publishing.")

items with at least one PubMed ID: 100%

20 'HABIT MISLEADS' ITEMS TO VERIFY BY HAND
For each: does the cited paper really show this direction of effect?

  1. PAX3 -> MBP   OmniPath says: INHIBITS
     (PAX3 usually stimulates; 2 refs; collectri)
     https://pubmed.ncbi.nlm.nih.gov/22532290/
     https://pubmed.ncbi.nlm.nih.gov/7546735/

  2. PTPN6 -> ABL1   OmniPath says: STIMULATES
     (PTPN6 usually inhibits; 6 refs; causal_ppi)
     https://pubmed.ncbi.nlm.nih.gov/10488096/
     https://pubmed.ncbi.nlm.nih.gov/10574931/
     https://pubmed.ncbi.nlm.nih.gov/12468540/

  3. DVL1 -> APC   OmniPath says: INHIBITS
     (DVL1 usually stimulates; 3 refs; causal_ppi)
     https://pubmed.ncbi.nlm.nih.gov/10330181/
     https://pubmed.ncbi.nlm.nih.gov/11425858/
     https://pubmed.ncbi.nlm.nih.gov/20224554/

  4. HDAC1 -> COL1A2   OmniPath says: STIMULATES
     (HDAC1 usually inhibits; 2 refs; collectri)
     https://pubmed.ncbi.nlm.nih.gov/16464847/
     https://pubmed.ncbi.nlm.nih.gov/2

In [20]:
# ============================================================================
# 04 — ANALYSIS
# ============================================================================
import json
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import binomtest, wilcoxon

ROOT = Path.cwd().parent
PROC, RES = ROOT/"data"/"processed", ROOT/"results"

A = pd.concat([pd.read_json(p, lines=True) for p in sorted(RES.glob("*.jsonl"))],
              ignore_index=True)
A = A[A.http==200].copy()
A["abstained"] = A.pred=="not_established"
A["committed"] = A.pred.notna() & ~A.abstained
A["correct"]   = A.pred.notna() & (A.pred==A.truth)
BENCH = pd.read_json(PROC/"sighabit_v2.jsonl", lines=True)

MAIN = A[(A.arm=="main") & (~A.is_decoy)]
DEC  = A[(A.arm=="main") &   A.is_decoy]
MODELS_ = sorted(MAIN.model.unique())
print(f"{len(A):,} rows | {len(MODELS_)} models | {MAIN.item_id.nunique()} questions "
      f"| {DEC.item_id.nunique()} decoys")

# ---- 1. is the gap knowledge, or refusal? --------------------------------
print("\n"+"="*94)
print("1. THE GAP — counting refusals as wrong, then ignoring them")
print("="*94)
print("If the gap survives when we score ONLY the answers a model committed to,")
print("it is a knowledge failure. If it vanishes, it is just extra caution.\n")
print(f"{'model':<20}{'agree':>7}{'mislead':>9}{'gap':>7}  |{'agree*':>8}{'mislead*':>10}"
      f"{'gap*':>7}  |{'abst_ag':>9}{'abst_mis':>10}")
print("-"*94)
for m in MODELS_:
    g = MAIN[MAIN.model==m]
    a, b = g[g.condition=="habit_agrees"], g[g.condition=="habit_misleads"]
    ac, bc = a[a.committed], b[b.committed]
    print(f"{m:<20}{a.correct.mean():>7.1%}{b.correct.mean():>9.1%}"
          f"{(a.correct.mean()-b.correct.mean())*100:>7.1f}  |"
          f"{ac.correct.mean():>8.1%}{bc.correct.mean():>10.1%}"
          f"{(ac.correct.mean()-bc.correct.mean())*100:>7.1f}  |"
          f"{a.abstained.mean():>9.1%}{b.abstained.mean():>10.1%}")
print("\n  * = committed answers only")

# ---- 2. does it hold in BOTH habit directions? ---------------------------
print("\n"+"="*94)
print("2. BOTH DIRECTIONS — the control that rules out 'models say activate too much'")
print("="*94)
print("For usually-inhibits proteins the exception is 'stimulates'. A blanket")
print("pro-activation bias would HELP there. If the gap appears anyway, the")
print("model is following each protein's own default.\n")
print(f"{'model':<20}{'usually stimulates':>26}{'usually inhibits':>24}")
print(f"{'':<20}{'agree':>9}{'mislead':>9}{'gap':>8}{'agree':>9}{'mislead':>9}{'gap':>8}")
print("-"*94)
for m in MODELS_:
    row = f"{m:<20}"
    for u in ["stimulates","inhibits"]:
        g = MAIN[(MAIN.model==m) & (MAIN.protein_usually==u)]
        a = g[g.condition=="habit_agrees"].correct.mean()
        b = g[g.condition=="habit_misleads"].correct.mean()
        row += f"{a:>9.1%}{b:>9.1%}{(a-b)*100:>8.1f}"
    print(row)
n_i = MAIN[MAIN.protein_usually=="inhibits"].item_id.nunique()
print(f"\n  n = {n_i} questions on usually-inhibits proteins")

# ---- 3. paired test, protein by protein ---------------------------------
print("\n"+"="*94)
print("3. PAIRED TEST — each protein is its own control")
print("="*94)
print("Same protein, two questions. Counting how often it gets the usual one")
print("right and the exception wrong, versus the other way round.\n")
print(f"{'model':<20}{'agree only':>12}{'mislead only':>14}{'both':>7}{'neither':>9}"
      f"{'p':>11}")
print("-"*94)
for m in MODELS_:
    g = MAIN[MAIN.model==m]
    p = g.pivot_table(index="source", columns="condition", values="correct",
                      aggfunc="first")
    if not {"habit_agrees","habit_misleads"} <= set(p.columns): continue
    p = p.dropna()
    a_only = int(( p.habit_agrees & ~p.habit_misleads).sum())
    b_only = int((~p.habit_agrees &  p.habit_misleads).sum())
    both   = int(( p.habit_agrees &  p.habit_misleads).sum())
    none   = int((~p.habit_agrees & ~p.habit_misleads).sum())
    pv = binomtest(a_only, a_only+b_only, 0.5).pvalue if a_only+b_only else 1
    print(f"{m:<20}{a_only:>12}{b_only:>14}{both:>7}{none:>9}{pv:>11.2e}")
print("\n  'agree only' >> 'mislead only' is the effect. McNemar exact test.")


# ---- 4. THE MECHANISM — whose habit is the model following? --------------
# The model and the database agree about a protein's habit ~85% of the time.
# When two predictors agree that often you cannot tell them apart, which is
# why the first version of this test was uninformative. So we look only at
# the proteins where they DISAGREE. There, following one habit means
# contradicting the other, and the model has to reveal which it uses.
print("\n"+"="*94)
print("4. THE MECHANISM — whose habit does the model follow?")
print("="*94)
U = A[A.arm=="usually"]

print("4a. How often does each model's stated habit match the database?\n")
print(f"{'model':<20}{'proteins':>10}{'agree':>9}{'disagree':>10}"
      f"{'said unknown':>14}")
print("-"*94)
DIS = {}
for m in MODELS_:
    u = U[U.model==m]
    said = u[u.pred.isin(["stimulates","inhibits"])]
    d = said[said.pred != said.truth]
    DIS[m] = set(d.source)
    print(f"{m:<20}{len(u):>10}{(said.pred==said.truth).mean():>9.1%}"
          f"{len(d):>10}{(~u.pred.isin(['stimulates','inhibits'])).mean():>14.1%}")

print("\n4b. On the DISAGREEMENT proteins — whose habit wins?\n")
print("For these proteins the model's own belief and the database point in")
print("OPPOSITE directions, so every answer follows one or the other.\n")
print(f"{'model':<20}{'proteins':>10}{'questions':>11}{'follows OWN':>13}"
      f"{'follows DB':>12}{'p':>10}")
print("-"*94)
mech = []
for m in MODELS_:
    u = U[U.model==m][["source","pred"]].rename(columns={"pred":"own_habit"})
    g = MAIN[(MAIN.model==m) & MAIN.source.isin(DIS[m]) & MAIN.committed]
    g = g.merge(u, on="source", how="left")
    g = g[g.own_habit.isin(["stimulates","inhibits"])]
    if len(g) < 8:
        print(f"{m:<20}{len(DIS[m]):>10}{len(g):>11}   too few to test")
        continue
    # db_habit is the opposite of own_habit on these proteins, by construction
    g["db_habit"] = np.where(g.own_habit=="stimulates","inhibits","stimulates")
    own = int((g.pred==g.own_habit).sum()); db = int((g.pred==g.db_habit).sum())
    pv  = binomtest(own, own+db, 0.5).pvalue if own+db else 1.0
    mech.append(dict(model=m, own=own, db=db, p=pv))
    print(f"{m:<20}{len(DIS[m]):>10}{len(g):>11}{own/(own+db):>13.1%}"
          f"{db/(own+db):>12.1%}{pv:>10.3f}")
print("\n  >50% 'follows OWN' means the shortcut lives in the model, not in")
print("  our database — that is the mechanism claim.")

print("\n4c. Does the model's own habit predict WHERE it fails?\n")
print("Split every question by whether the truth matches the model's OWN")
print("stated belief about that protein. If its belief drives its answers,")
print("accuracy should collapse when the truth contradicts it.\n")
print(f"{'model':<20}{'truth matches':>15}{'truth contradicts':>19}{'gap':>8}"
      f"{'n_contra':>10}")
print("-"*94)
for m in MODELS_:
    u = U[U.model==m][["source","pred"]].rename(columns={"pred":"own_habit"})
    g = MAIN[MAIN.model==m].merge(u, on="source", how="left")
    g = g[g.own_habit.isin(["stimulates","inhibits"])]
    match = g[g.truth==g.own_habit]; contra = g[g.truth!=g.own_habit]
    if not len(contra): continue
    print(f"{m:<20}{match.correct.mean():>15.1%}{contra.correct.mean():>19.1%}"
          f"{(match.correct.mean()-contra.correct.mean())*100:>8.1f}"
          f"{len(contra):>10}")
print("\n  Compare this gap to the database-habit gap in section 1. If it is")
print("  as large or larger, the model's own belief is the better predictor.")

print("\n4d. Head to head — which habit predicts errors better?\n")
print(f"{'model':<20}{'DB-habit gap':>14}{'own-habit gap':>15}{'difference':>12}")
print("-"*94)
for m in MODELS_:
    g0 = MAIN[MAIN.model==m]
    db_gap = (g0[g0.condition=="habit_agrees"].correct.mean()
              - g0[g0.condition=="habit_misleads"].correct.mean())*100
    u = U[U.model==m][["source","pred"]].rename(columns={"pred":"own_habit"})
    g = g0.merge(u, on="source", how="left")
    g = g[g.own_habit.isin(["stimulates","inhibits"])]
    own_gap = (g[g.truth==g.own_habit].correct.mean()
               - g[g.truth!=g.own_habit].correct.mean())*100
    print(f"{m:<20}{db_gap:>14.1f}{own_gap:>15.1f}{own_gap-db_gap:>+12.1f}")
print("\n  A positive difference means the model's OWN belief explains its")
print("  errors better than our database does.")
# ---- 5. option order ------------------------------------------------------
print("\n"+"="*94); print("5. OPTION ORDER — validity check"); print("="*94)
SW = A[A.arm=="swap"]
print(f"{'model':<20}{'says inhibits (normal)':>24}{'(swapped)':>12}{'shift':>8}")
print("-"*94)
for m in sorted(SW.model.unique()):
    s = SW[SW.model==m]
    n = MAIN[(MAIN.model==m) & MAIN.item_id.isin(set(s.item_id))]
    a, b = (n.pred=="inhibits").mean(), (s.pred=="inhibits").mean()
    print(f"{m:<20}{a:>24.1%}{b:>12.1%}{(b-a)*100:>+8.1f}")
missing = set(MODELS_) - set(SW.model.unique())
print(f"\n  NOT TESTED: {sorted(missing)}" if missing else "")
print("  A shift above ~20 points means the model answers from list order.")

# ---- 6. search ------------------------------------------------------------
print("\n"+"="*94); print("6. DOES RETRIEVAL FIX IT?"); print("="*94)
W = A[A.arm=="web"]
if len(W):
    m = W.model.iloc[0]
    base = MAIN[(MAIN.model==m) & MAIN.item_id.isin(set(W.item_id))]
    print(f"{'condition':<18}{'no search':>12}{'web search':>12}{'change':>9}")
    print("-"*54)
    for c in ["habit_agrees","habit_misleads"]:
        a, b = base[base.condition==c].correct.mean(), W[W.condition==c].correct.mean()
        print(f"{c:<18}{a:>12.1%}{b:>12.1%}{(b-a)*100:>+9.1f}")
    ga = (base[base.condition=='habit_agrees'].correct.mean()
          - base[base.condition=='habit_misleads'].correct.mean())*100
    gw = (W[W.condition=='habit_agrees'].correct.mean()
          - W[W.condition=='habit_misleads'].correct.mean())*100
    print(f"\n  GAP: {ga:.1f} pp without search  ->  {gw:.1f} pp with search")
    print("  Unchanged means retrieval raises the level without changing the strategy.")

# ---- 7. fabricated pairs --------------------------------------------------
print("\n"+"="*94); print("7. FABRICATED PAIRS"); print("="*94)
print(f"{'model':<20}{'refused fake':>14}{'abstained real':>16}{'separation':>12}")
print("-"*94)
for m in MODELS_:
    d, r = DEC[DEC.model==m], MAIN[MAIN.model==m]
    print(f"{m:<20}{d.abstained.mean():>14.1%}{r.abstained.mean():>16.1%}"
          f"{(d.abstained.mean()-r.abstained.mean())*100:>+12.1f}")
print("\n  Near-zero refusal means the model invents interactions.")

# ---- 8. all models, one direction? ---------------------------------------
print("\n"+"="*94); print("8. SUMMARY"); print("="*94)
gaps = {m: (MAIN[(MAIN.model==m)&(MAIN.condition=="habit_agrees")].correct.mean()
          - MAIN[(MAIN.model==m)&(MAIN.condition=="habit_misleads")].correct.mean())*100
        for m in MODELS_}
k = sum(v>0 for v in gaps.values())
print(f"  gap positive in {k}/{len(gaps)} models   "
      f"p = {binomtest(k, len(gaps), 0.5).pvalue:.4f}")
print(f"  median gap {np.median(list(gaps.values())):.1f} pp   "
      f"range {min(gaps.values()):.1f} to {max(gaps.values()):.1f}")
below = [m for m in MODELS_
         if MAIN[(MAIN.model==m)&(MAIN.condition=='habit_misleads')].correct.mean() < .5]
print(f"  below chance on the exceptions: {len(below)}/{len(MODELS_)} models")

3,516 rows | 5 models | 216 questions | 120 decoys

1. THE GAP — counting refusals as wrong, then ignoring them
If the gap survives when we score ONLY the answers a model committed to,
it is a knowledge failure. If it vanishes, it is just extra caution.

model                 agree  mislead    gap  |  agree*  mislead*   gap*  |  abst_ag  abst_mis
----------------------------------------------------------------------------------------------
gemini-3.7-flash      68.5%    37.0%   31.5  |   88.1%     51.9%   36.1  |    22.2%     28.7%
gpt-5.6-luna          65.7%    32.4%   33.3  |   86.6%     41.2%   45.4  |    24.1%     21.3%
gpt-5.6-terra         71.3%    43.5%   27.8  |   86.5%     50.0%   36.5  |    17.6%     13.0%
gpt-oss-120b          50.0%    30.6%   19.4  |   90.0%     50.8%   39.2  |    44.4%     39.8%
qwen3-32b             42.6%    16.7%   25.9  |   86.8%     37.5%   49.3  |    50.9%     55.6%

  * = committed answers only

2. BOTH DIRECTIONS — the control that rules out 'models

In [22]:
# ============================================================================
# 05b — LABEL AUDIT, WITH ALIASES
# ----------------------------------------------------------------------------
# The first version gave modern gene symbols; the papers use 1980s-90s names
# (TCF4 is "E2-2", KDR is "VEGFR2"). Nobody can audit 40 items while doing
# that translation by hand. This looks up every synonym from UniProt and puts
# them on the sheet.
# ============================================================================
import json, urllib.request, urllib.parse, time
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
RAW, PROC, DOCS = ROOT/"data"/"raw", ROOT/"data"/"processed", ROOT/"docs"
DOCS.mkdir(exist_ok=True)

B = pd.read_json(PROC/"sighabit_v2.jsonl", lines=True)

# ---- references behind each edge ----------------------------------------
ref_map = {}
for n in ["causal_ppi","collectri","kinaseextra"]:
    d = pd.read_parquet(RAW/f"{n}.parquet")
    if "references" not in d.columns: continue
    for s,t,r in zip(d.source_genesymbol, d.target_genesymbol, d["references"]):
        if isinstance(r,str) and r.strip():
            ref_map.setdefault((s,t), set()).update(
                x.split(":")[-1] for x in r.split(";")
                if x.strip().split(":")[-1].isdigit())
B["pmids"] = [sorted(ref_map.get((r.source,r.target), set()))[:3] for _,r in B.iterrows()]

# ---- every alias, from UniProt ------------------------------------------
cache = PROC/"gene_aliases.json"
if cache.exists():
    ALIAS = json.loads(cache.read_text()); print(f"aliases cached ({len(ALIAS)})")
else:
    acc = {}
    for _, r in B.iterrows():
        if isinstance(r.source_uniprot,str): acc.setdefault(r.source, r.source_uniprot)
        if isinstance(r.target_uniprot,str): acc.setdefault(r.target, r.target_uniprot)
    a2g = {v:k for k,v in acc.items()}
    accs, ALIAS = sorted(acc.values()), {}
    print(f"fetching synonyms for {len(accs)} proteins...")
    for i in range(0, len(accs), 50):
        q = "(" + " OR ".join(f"accession:{a}" for a in accs[i:i+50]) + ")"
        try:
            u = ("https://rest.uniprot.org/uniprotkb/stream?format=tsv"
                 f"&fields=accession,gene_names,protein_name&query={urllib.parse.quote(q)}")
            txt = urllib.request.urlopen(u, timeout=120).read().decode()
            for ln in txt.splitlines()[1:]:
                p = ln.split("\t")
                if len(p) >= 2 and p[0] in a2g:
                    g = a2g[p[0]]
                    syn = [x for x in p[1].split() if x.upper() != g.upper()]
                    ALIAS[g] = dict(aliases=syn[:5],
                                    name=(p[2][:60] if len(p) > 2 else ""))
        except Exception as e:
            print(f"  batch {i} failed: {type(e).__name__}")
        time.sleep(0.2)
    cache.write_text(json.dumps(ALIAS)); print(f"  got {len(ALIAS)}")

def names(g):
    a = ALIAS.get(g, {}).get("aliases", [])
    return f"**{g}**" + (f" *(also: {', '.join(a)})*" if a else "")

# ---- sample, blinded -----------------------------------------------------
rng = np.random.RandomState(42)
S = pd.concat([
    B[(B.condition=="habit_misleads") & (B.pmids.map(len)>0)].sample(20, random_state=1),
    B[(B.condition=="habit_agrees")   & (B.pmids.map(len)>0)].sample(20, random_state=2)])
S = S.iloc[rng.permutation(len(S))].reset_index(drop=True)
S["audit_id"] = [f"A{i+1:02d}" for i in range(len(S))]

L = ["# Label audit — blinded", "",
"## How to judge each one", "",
"Open the papers. Ask: **does this paper show that the first protein",
"decreases or increases the activity or level of the second?**", "",
"| Mark | When |",
"|------|------|",
"| **Y** | the paper supports the stated direction |",
"| **N** | the paper shows the OPPOSITE direction |",
"| **?** | ambiguous, context-dependent, or the paper never addresses this pair |",
"| **X** | the paper is not about these two proteins at all |", "",
"### Reading tips", "",
"- Papers often use OLD names. The aliases are listed for you — search the",
"  abstract for any of them.",
"- 'represses / suppresses / blocks / downregulates' = **inhibits**",
"- 'induces / activates / upregulates / promotes / required for' = **stimulates**",
"- Promoter or reporter assays count: repressing a promoter is inhibition.",
"- If the abstract is not enough, the full text usually is — but if you cannot",
"  tell in ~2 minutes, mark **?** and move on. '?' is a real answer.", "",
"**Do not look at the answer key until you have finished all 40.**", "",
"---", ""]

for _, r in S.iterrows():
    L += [f"### {r.audit_id}", "",
          f"**Claim:** {r.source} → {r.target} — OmniPath says "
          f"**{r.answer.upper()}**", "",
          f"- upstream: {names(r.source)}",
          f"  <br>*{ALIAS.get(r.source,{}).get('name','')}*",
          f"- downstream: {names(r.target)}",
          f"  <br>*{ALIAS.get(r.target,{}).get('name','')}*", "",
          "**Papers:** " + "  ".join(
              f"[{p}](https://pubmed.ncbi.nlm.nih.gov/{p}/)" for p in r.pmids), "",
          "**Verdict:** `   `  (Y / N / ? / X)", "",
          "**Note:** ", "", "---", ""]

(DOCS/"label_audit_blinded.md").write_text("\n".join(L), encoding="utf-8")
S[["audit_id","source","target","answer","condition","protein_usually",
   "n_refs","component"]].to_csv(DOCS/"label_audit_key.csv", index=False)

print(f"\nworksheet: {DOCS/'label_audit_blinded.md'}  ({len(S)} items)")
print(f"key      : {DOCS/'label_audit_key.csv'}  — do not open yet")
print("\nExample of the new format:")
r = S.iloc[0]
print(f"\n  Claim: {r.source} -> {r.target}, OmniPath says {r.answer.upper()}")
print(f"  upstream  : {names(r.source)}")
print(f"  downstream: {names(r.target)}")

fetching synonyms for 275 proteins...
  got 275

worksheet: c:\Users\tauhe\Desktop\SigHabitBench\docs\label_audit_blinded.md  (40 items)
key      : c:\Users\tauhe\Desktop\SigHabitBench\docs\label_audit_key.csv  — do not open yet

Example of the new format:

  Claim: TCF4 -> KDR, OmniPath says INHIBITS
  upstream  : **TCF4** *(also: BHLHB19, ITF2, SEF2)*
  downstream: **KDR** *(also: FLK1, VEGFR2)*


In [23]:
# ============================================================================
# 05c — UNBLIND THE AUDIT
# ----------------------------------------------------------------------------
# The only question that matters: are label errors spread evenly across both
# conditions, or concentrated in the "exception" items? If concentrated, part
# of our effect is database error rather than model behaviour.
# ============================================================================
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import fisher_exact

ROOT = Path.cwd().parent
DOCS = ROOT/"docs"

# adjudicated verdicts (A06, A20, A01 corrected — see notes)
VERDICT = {
 "A01":"Y","A02":"Y","A03":"Y","A04":"?","A05":"?","A06":"Y","A07":"?",
 "A08":"Y","A09":"?","A10":"Y","A11":"?","A12":"Y","A13":"Y","A14":"Y",
 "A15":"Y","A16":"N","A17":"?","A18":"?","A19":"Y","A20":"Y",
}
NOTE = {
 "A01":"E2-2 = TCF4; alias missing from worksheet. Papers support repression.",
 "A05":"paper attributes stimulation to an orphan receptor, not ESR1 directly",
 "A06":"'Gli2 is a negative regulator of type X collagen' = supports inhibits",
 "A15":"two papers conflict; counted Y on the stronger one",
 "A16":"paper says LCK UPREGULATES FOXP3 — genuine label error",
 "A18":"references include a 1987 paper predating the claim",
 "A20":"'Stimulation of SIRT1 transcription by Foxo3a' = supports stimulates",
}

KEY = pd.read_csv(DOCS/"label_audit_key.csv")
KEY["verdict"] = KEY.audit_id.map(VERDICT)
D = KEY.dropna(subset=["verdict"]).copy()
print(f"audited {len(D)} of {len(KEY)} items\n")

print("="*88); print("1. WHAT WAS IN THE SAMPLE"); print("="*88)
print(pd.crosstab(D.condition, D.verdict).to_string())

print("\n"+"="*88); print("2. THE DECIDING NUMBER"); print("="*88)
print("Among items where a verdict was reachable (Y or N), how often was the")
print("label right — in each condition?\n")
V = D[D.verdict.isin(["Y","N"])]
print(f"{'condition':<20}{'verified':>10}{'correct':>10}{'wrong':>8}{'accuracy':>11}")
print("-"*60)
for c in ["habit_agrees","habit_misleads"]:
    s = V[V.condition==c]
    if not len(s): print(f"{c:<20}{'none in sample':>39}"); continue
    print(f"{c:<20}{len(s):>10}{int((s.verdict=='Y').sum()):>10}"
          f"{int((s.verdict=='N').sum()):>8}{(s.verdict=='Y').mean():>11.0%}")
print(f"{'OVERALL':<20}{len(V):>10}{int((V.verdict=='Y').sum()):>10}"
      f"{int((V.verdict=='N').sum()):>8}{(V.verdict=='Y').mean():>11.0%}")

a = V[V.condition=="habit_agrees"]; b = V[V.condition=="habit_misleads"]
if len(a) and len(b):
    tab = [[int((a.verdict=="Y").sum()), int((a.verdict=="N").sum())],
           [int((b.verdict=="Y").sum()), int((b.verdict=="N").sum())]]
    p = fisher_exact(tab)[1]
    print(f"\n  Fisher exact p = {p:.3f}")
    print("  p > 0.05 = no evidence errors cluster in the exceptions")

print("\n"+"="*88); print("3. HOW OFTEN COULD WE NOT TELL?"); print("="*88)
print(f"{'condition':<20}{'n':>6}{'unverifiable':>14}")
print("-"*42)
for c in ["habit_agrees","habit_misleads"]:
    s = D[D.condition==c]
    if len(s):
        print(f"{c:<20}{len(s):>6}{s.verdict.isin(['?','X']).mean():>14.0%}")
print(f"{'OVERALL':<20}{len(D):>6}{D.verdict.isin(['?','X']).mean():>14.0%}")
print("\n  Unverifiable = the cited papers do not clearly address the pair.")
print("  Not the same as wrong, but worth reporting as a limitation.")

print("\n"+"="*88); print("4. EVERY ITEM, UNBLINDED"); print("="*88)
print(f"{'ID':<5}{'interaction':<24}{'label':<12}{'condition':<16}{'verdict':<9}note")
print("-"*100)
for _, r in D.sort_values("audit_id").iterrows():
    print(f"{r.audit_id:<5}{r.source+' -> '+r.target:<24}{r.answer:<12}"
          f"{r.condition:<16}{r.verdict:<9}{NOTE.get(r.audit_id,'')[:44]}")

print("\n"+"="*88); print("5. WHAT TO WRITE IN THE PAPER"); print("="*88)
acc = (V.verdict=="Y").mean(); unv = D.verdict.isin(["?","X"]).mean()
ga = (a.verdict=="Y").mean() if len(a) else np.nan
gb = (b.verdict=="Y").mean() if len(b) else np.nan
print(f'  "We manually audited {len(D)} randomly sampled interactions against')
print(f'   their cited primary literature, blinded to condition. Of the')
print(f'   {len(V)} for which the abstract permitted a verdict, {acc:.0%} were')
print(f'   supported ({ga:.0%} habit-consistent, {gb:.0%} habit-contradicting;')
print(f'   Fisher exact p = {p:.2f}). {unv:.0%} could not be verified from the')
print(f'   cited abstracts. We therefore find no evidence that label error is')
print(f'   concentrated in the habit-contradicting condition."')
if not np.isnan(ga) and not np.isnan(gb) and abs(ga-gb) > 0.25:
    print("\n  *** WARNING: the two conditions differ by more than 25 points.")
    print("      Audit the remaining 20 items before drawing conclusions. ***")

audited 20 of 40 items

1. WHAT WAS IN THE SAMPLE
verdict         ?  N  Y
condition              
habit_agrees    5  0  4
habit_misleads  2  1  8

2. THE DECIDING NUMBER
Among items where a verdict was reachable (Y or N), how often was the
label right — in each condition?

condition             verified   correct   wrong   accuracy
------------------------------------------------------------
habit_agrees                 4         4       0       100%
habit_misleads               9         8       1        89%
OVERALL                     13        12       1        92%

  Fisher exact p = 1.000
  p > 0.05 = no evidence errors cluster in the exceptions

3. HOW OFTEN COULD WE NOT TELL?
condition                n  unverifiable
------------------------------------------
habit_agrees             9           56%
habit_misleads          11           18%
OVERALL                 20           35%

  Unverifiable = the cited papers do not clearly address the pair.
  Not the same as wrong, but wort